# Day 12 — OSD-536 study record 읽기: 데이터를 열기 전에 설계부터

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-2/day-12.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-2/day-12.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — API로 파일 목록 확인하기

In [ ]:
import json
import urllib.request

url = "https://osdr.nasa.gov/osdr/data/osd/files/536"
with urllib.request.urlopen(url, timeout=60) as r:
    data = json.load(r)

study = data["studies"]["OSD-536"]
print("파일 수:", study["file_count"])
for f in study["study_files"]:
    print(f["category"], "|", f["file_name"], "|", f["file_size"], "bytes")

### 예시 2 — ISA 메타데이터에서 샘플 표 읽기

In [ ]:
import io
import zipfile
import urllib.request
import pandas as pd

isa_url = ("https://osdr.nasa.gov/geode-py/ws/studies/OSD-536/download"
           "?source=datamanager&file=OSD-536_metadata_OSD-536-ISA.zip")
with urllib.request.urlopen(isa_url, timeout=60) as r:
    z = zipfile.ZipFile(io.BytesIO(r.read()))

print(z.namelist())                                   # zip 안에 든 파일 이름
sample_file = [n for n in z.namelist() if n.startswith("s_")][0]
samples = pd.read_csv(z.open(sample_file), sep="\t")

print(samples.shape)
print(samples[["Source Name", "Sample Name", "Factor Value[Spaceflight]"]])
print(samples["Factor Value[Spaceflight]"].value_counts())

### 예시 3 — 열을 세 종류로 나누고 값의 종류 보기

예시 2를 실행한 같은 노트북에서 이어서 실행합니다.

In [ ]:
keep = [c for c in samples.columns
        if c.startswith(("Characteristics", "Factor Value", "Parameter Value"))]

for c in keep:
    values = samples[c].astype(str).str.strip().unique()
    print(f"{c}: {list(values)}")

## 직접 해보기

1. 브라우저에서 <https://osdr.nasa.gov/bio/repo/data/studies/OSD-536>을 엽니다. 페이지의 Description(설명)을 끝까지 읽고, 연구 목적을 한국어 두 문장으로 요약해 둡니다.
2. 페이지에서 샘플 표, assay 표, 프로토콜, 파일 목록에 해당하는 탭이나 섹션을 찾아 한 번씩 열어 봅니다. (화면 구성은 바뀔 수 있습니다. 이름이 조금 달라도 같은 내용을 찾으면 됩니다.)
3. 새 노트북 `notebooks/01_setup_and_metadata.ipynb`를 열고(1주차에 만들었다면 맨 아래에 `## Day 12 — OSD-536 메타데이터` 섹션 추가) 예시 1, 2, 3을 실행합니다. 출력이 이 레슨과 같은지 확인하고, **다르면 다른 부분과 조회 날짜를 적어 둡니다.** 다르다고 해서 여러분이 틀린 것이 아닙니다. 데이터베이스가 갱신되었을 수 있습니다.
4. 프로토콜 글을 읽습니다. 웹 페이지의 프로토콜 섹션에서 읽거나, 노트북에서 다음 코드로 Investigation 파일의 프로토콜 설명을 꺼냅니다(예시 2의 `z`를 이어서 씁니다).

In [ ]:
text = z.read("i_Investigation.txt").decode("utf-8", errors="replace")
names, descs = [], []
for line in text.splitlines():
    if line.startswith("Study Protocol Name"):
        names = line.split("\t")[1:]          # 프로토콜 이름들이 탭으로 나뉘어 한 줄에 있다
    if line.startswith("Study Protocol Description"):
        descs = line.split("\t")[1:]

for name, desc in zip(names, descs):
    print(f"[{name}]")
    print(desc)
    print()